# H1 / H1b — Has the frequency of accusations of lying changed over time?

**H1**  the frequency of accusations of lying has increased over time.
**H1b** (competing) it has remained stable.

### Design

The unit is the **country-year**. The outcome is a *rate*: accusations divided by
sentences actually spoken, so a country that simply enters the corpus with more
text does not look like it accuses more.

Main test: Poisson regression of the accusation count on year, with
`offset(log n_sentences)` and **country fixed effects**, cluster-robust SEs by
country. The year coefficient is the average *within-country* log-linear trend —
which is precisely the H1-vs-H1b contrast.

Three things could manufacture a spurious trend, so each gets a robustness check
at the bottom: changing country composition, changing source/translation quality,
and the classifier threshold.


In [ ]:
import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from lib import data, viz
viz.apply_style()

MIN_SENTENCES = 10_000   # country-years below this are too sparse for a stable rate

# Analysis window. Before 1994 only 1-3 parliaments are in the corpus, and after
# 2022 coverage collapses as corpora end at different dates -- both ends produce
# rates driven by composition rather than behaviour. Applied to the plots AND the
# models so they always describe the same data.
YEAR_MIN, YEAR_MAX = 1994, 2022

## 1. Country-year rates

Numerator from the accusation dataset, denominator from the full corpus. Both
views already exclude IS / BA / GR / LV.

In [ ]:
con = data.duck()

cy = con.execute("""
WITH sents AS (
    SELECT country,
           CAST(substr(date, 1, 4) AS INT) AS year,
           COUNT(*) AS n_sentences
    FROM corpus
    WHERE date IS NOT NULL AND length(date) >= 4
    GROUP BY 1, 2
), accs AS (
    SELECT country,
           CAST(substr(date, 1, 4) AS INT) AS year,
           COUNT(*) AS n_accusations
    FROM accusations
    WHERE date IS NOT NULL AND length(date) >= 4
    GROUP BY 1, 2
)
SELECT s.country, s.year, s.n_sentences,
       COALESCE(a.n_accusations, 0) AS n_accusations
FROM sents s
LEFT JOIN accs a USING (country, year)
ORDER BY country, year
""").df()

cy["rate_10k"] = cy["n_accusations"] / cy["n_sentences"] * 10_000

print(f"{cy['country'].nunique()} countries, {cy['year'].min()}-{cy['year'].max()}, "
      f"{len(cy):,} country-years")
print(f"total sentences   : {cy['n_sentences'].sum():,}")
print(f"total accusations : {cy['n_accusations'].sum():,}")
cy.head()

In [ ]:
cyf = cy[(cy["n_sentences"] >= MIN_SENTENCES)
         & cy["year"].between(YEAR_MIN, YEAR_MAX)].copy()

dropped_thin = (cy["n_sentences"] < MIN_SENTENCES).sum()
dropped_year = (~cy["year"].between(YEAR_MIN, YEAR_MAX)).sum()
print(f"country-years: {len(cy):,} -> {len(cyf):,}")
print(f"  dropped, < {MIN_SENTENCES:,} sentences : {dropped_thin:,}")
print(f"  dropped, outside {YEAR_MIN}-{YEAR_MAX}   : {dropped_year:,}")
print(f"     {cyf['n_sentences'].sum() / cy['n_sentences'].sum() * 100:.1f}% of all sentences kept")
print(f"     {cyf['country'].nunique()} countries, {cyf['year'].min()}-{cyf['year'].max()}")

# how many years does each country contribute?
span = (cyf.groupby("country")
           .agg(years=("year", "nunique"), first=("year", "min"), last=("year", "max"),
                sentences=("n_sentences", "sum"))
           .sort_values("years", ascending=False))
span

## 2. Pooled trend (descriptive)

Restricted to the `YEAR_MIN`–`YEAR_MAX` window set at the top (default
**1994–2022**). Outside it the corpus holds only a handful of parliaments, so the
rate reflects *which* countries are present rather than how often anyone accuses
anyone.

The grey bars show how many countries are in the corpus each year. **Read the
line with the bars in mind** — even inside the window, composition shifts. The
fixed-effects model in section 4 is what actually tests H1.

In [ ]:
g = cyf.groupby("year")
pooled = pd.DataFrame({
    "n_accusations": g["n_accusations"].sum(),
    "n_sentences":   g["n_sentences"].sum(),
    "n_countries":   g["country"].nunique(),
}).reset_index()
pooled["rate_10k"] = pooled["n_accusations"] / pooled["n_sentences"] * 10_000

fig, ax = plt.subplots(figsize=(9, 5))
ax2 = ax.twinx()
ax2.bar(pooled["year"], pooled["n_countries"], alpha=0.15, color="grey", zorder=1)
ax2.set_ylabel("countries in corpus (bars)")
ax2.set_zorder(1)
ax.plot(pooled["year"], pooled["rate_10k"], marker="o", ms=3, zorder=3)
ax.set_zorder(2); ax.patch.set_visible(False)
ax.set_xlabel("year")
ax.set_ylabel("accusations per 10,000 sentences")
ax.set_title("Accusation rate over time (pooled across countries)")
viz.savefig(fig, "h1_pooled_rate")
plt.show()

pooled.tail(15)

### Monthly version, with corpus volume behind it

Same rate, at monthly resolution, with the **volume of the corpus** as a shaded
area behind it. This makes visible how much text each point rests on — thin
stretches are where the rate is least trustworthy — and shows the parliamentary
calendar directly (the regular troughs are recesses).

Thin grey line = raw monthly rate; heavy line = 12-month centred average.

In [ ]:
mon = con.execute(f"""
WITH sents AS (
    SELECT substr(date, 1, 7) AS ym, COUNT(*) AS n_sentences
    FROM corpus
    WHERE date IS NOT NULL AND length(date) >= 7
      AND CAST(substr(date, 1, 4) AS INT) BETWEEN {YEAR_MIN} AND {YEAR_MAX}
    GROUP BY 1
), accs AS (
    SELECT substr(date, 1, 7) AS ym, COUNT(*) AS n_accusations
    FROM accusations
    WHERE date IS NOT NULL AND length(date) >= 7
      AND CAST(substr(date, 1, 4) AS INT) BETWEEN {YEAR_MIN} AND {YEAR_MAX}
    GROUP BY 1
)
SELECT s.ym, s.n_sentences, COALESCE(a.n_accusations, 0) AS n_accusations
FROM sents s LEFT JOIN accs a USING (ym)
ORDER BY s.ym
""").df()

mon = mon[mon["ym"].str.len() == 7].copy()
mon["date"] = pd.to_datetime(mon["ym"] + "-01", errors="coerce")
mon = mon.dropna(subset=["date"]).sort_values("date")
mon["rate_10k"] = mon["n_accusations"] / mon["n_sentences"] * 10_000
mon["rate_smooth"] = (mon["rate_10k"]
                      .rolling(12, center=True, min_periods=6).mean())

print(f"{len(mon):,} months, {mon['date'].min():%Y-%m} to {mon['date'].max():%Y-%m}")
print(f"median sentences/month: {mon['n_sentences'].median():,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
axv = ax.twinx()

# corpus volume behind everything
axv.fill_between(mon["date"], mon["n_sentences"] / 1e6,
                 color="#8ec5e8", alpha=0.45, lw=0, zorder=1)
axv.set_ylabel("million sentences per month", color="#3a7ca8")
axv.tick_params(axis="y", colors="#3a7ca8")
axv.set_ylim(bottom=0)

# rate on top
ax.plot(mon["date"], mon["rate_10k"], lw=0.7, color="#8a8a8a", alpha=0.8,
        zorder=2, label="monthly rate")
ax.plot(mon["date"], mon["rate_smooth"], lw=2.2, color="#1f4e79",
        zorder=3, label="12-month average")
ax.set_ylabel("accusations per 10,000 sentences")
ax.set_xlabel("year")
ax.set_ylim(bottom=0)

# put the rate axis in front of the shaded area
ax.set_zorder(axv.get_zorder() + 1)
ax.patch.set_visible(False)

ax.set_title("Accusation rate and corpus volume over time")
ax.legend(loc="upper left", frameon=True)
fig.tight_layout()
viz.savefig(fig, "h1_rate_with_volume")
plt.show()

## 3. Per-country trends

Each panel is one country. This is the honest picture: H1 predicts most panels
slope upward.

In [ ]:
countries = sorted(cyf["country"].unique())
ncol = 5
nrow = int(np.ceil(len(countries) / ncol))

fig, axes = plt.subplots(nrow, ncol, figsize=(16, 2.4 * nrow), sharex=True)
for ax, c in zip(axes.flat, countries):
    d = cyf[cyf["country"] == c].sort_values("year")
    ax.plot(d["year"], d["rate_10k"], lw=1.2)
    if len(d) >= 3:                      # light OLS guide line
        z = np.polyfit(d["year"], d["rate_10k"], 1)
        ax.plot(d["year"], np.polyval(z, d["year"]), lw=1, ls="--", color="grey")
    ax.set_title(f"{c}  (n={len(d)})", fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes.flat[len(countries):]:
    ax.axis("off")
fig.suptitle("Accusations per 10,000 sentences, by country "
             "(dashed = linear fit)", y=1.002)
fig.tight_layout()
viz.savefig(fig, "h1_country_trends")
plt.show()

## 4. Main test — Poisson, country FE, exposure offset

$n\_accusations_{ct} \sim \text{Poisson}$, $\;\log$ link,
offset $\log(n\_sentences_{ct})$, country fixed effects, year centred.
SEs clustered by country.

- year coefficient **> 0 and significant** → supports **H1**
- year coefficient **≈ 0 with a tight CI** → supports **H1b**

In [ ]:
cyf["year_c"] = cyf["year"] - cyf["year"].mean()

pois = (smf.glm("n_accusations ~ year_c + C(country)", data=cyf,
                family=sm.families.Poisson(),
                offset=np.log(cyf["n_sentences"]))
           .fit(cov_type="cluster", cov_kwds={"groups": cyf["country"]}))

def report(m, label, term="year_c"):
    b = m.params[term]
    lo, hi = m.conf_int().loc[term]
    print(f"{label:<34} b={b:+.4f}  [{lo:+.4f}, {hi:+.4f}]  "
          f"p={m.pvalues[term]:.3g}  "
          f"{(np.exp(b * 10) - 1) * 100:+6.1f}% / decade")
    return b

print(f"n country-years: {int(pois.nobs):,}   countries: {cyf['country'].nunique()}\n")
report(pois, "Poisson, country FE")

### Overdispersion

Poisson assumes variance = mean. Count data at this scale rarely obliges; if the
dispersion statistic is far above 1 the Poisson SEs are too small and the
negative binomial below is the one to report.

In [ ]:
disp = pois.pearson_chi2 / pois.df_resid
print(f"Pearson dispersion: {disp:,.1f}   (1.0 = Poisson assumption holds)")

nb = (smf.glm("n_accusations ~ year_c + C(country)", data=cyf,
              family=sm.families.NegativeBinomial(alpha=1.0),
              offset=np.log(cyf["n_sentences"]))
         .fit(cov_type="cluster", cov_kwds={"groups": cyf["country"]}))

report(pois, "Poisson, country FE")
report(nb,   "Negative binomial, country FE")
print("\nIf these two disagree materially, report the negative binomial.")

## 5. Robustness

Each check targets one way the headline trend could be an artefact.

### 5a. Balanced panel — changing country composition

Countries enter and leave the corpus. Restricting to those observed over a long
span removes composition as an explanation.

In [ ]:
results = {}
results["all country-years"] = report(pois, "all country-years")

for min_years in (10, 15, 20):
    keep = span[span["years"] >= min_years].index
    sub = cyf[cyf["country"].isin(keep)].copy()
    if sub["country"].nunique() < 3:
        print(f"(skipped >= {min_years} yrs: only {sub['country'].nunique()} countries)")
        continue
    m = (smf.glm("n_accusations ~ year_c + C(country)", data=sub,
                 family=sm.families.Poisson(),
                 offset=np.log(sub["n_sentences"]))
            .fit(cov_type="cluster", cov_kwds={"groups": sub["country"]}))
    results[f">= {min_years} yrs"] = report(
        m, f">= {min_years} yrs ({sub['country'].nunique()} countries)")

### 5b. Source fixed effects — translation / digitisation quality

Some corpora are machine-translated and some are OCR'd; both have improved over
time, which could look like more accusations. Swapping country FE for
**source_dataset FE** absorbs differences between sources.

In [ ]:
ds = con.execute("""
WITH sents AS (
    SELECT source_dataset, country,
           CAST(substr(date, 1, 4) AS INT) AS year,
           COUNT(*) AS n_sentences
    FROM corpus
    WHERE date IS NOT NULL AND length(date) >= 4
    GROUP BY 1, 2, 3
), accs AS (
    SELECT source_dataset, country,
           CAST(substr(date, 1, 4) AS INT) AS year,
           COUNT(*) AS n_accusations
    FROM accusations
    WHERE date IS NOT NULL AND length(date) >= 4
    GROUP BY 1, 2, 3
)
SELECT s.*, COALESCE(a.n_accusations, 0) AS n_accusations
FROM sents s LEFT JOIN accs a USING (source_dataset, country, year)
""").df()

ds = ds[ds["n_sentences"] >= MIN_SENTENCES].copy()
ds["year_c"] = ds["year"] - cyf["year"].mean()
print(f"{len(ds):,} dataset-country-years, {ds['source_dataset'].nunique()} datasets\n")

m_ds = (smf.glm("n_accusations ~ year_c + C(source_dataset)", data=ds,
                family=sm.families.Poisson(),
                offset=np.log(ds["n_sentences"]))
           .fit(cov_type="cluster", cov_kwds={"groups": ds["source_dataset"]}))

report(pois, "country FE (main)")
report(m_ds, "source_dataset FE")

### 5c. Shape of the trend — when, if ever, does it move?

A single linear slope hides timing. Year dummies (country FE, same offset) show
the year-by-year profile relative to the first year. A post-2010 jump would fit
the post-truth narrative; a steady drift or flat line would not.

In [ ]:
m_yr = (smf.glm("n_accusations ~ C(year) + C(country)", data=cyf,
                family=sm.families.Poisson(),
                offset=np.log(cyf["n_sentences"]))
           .fit(cov_type="cluster", cov_kwds={"groups": cyf["country"]}))

rows = []
for term in m_yr.params.index:
    if term.startswith("C(year)[T."):
        yr = int(term.split("T.")[1].rstrip("]"))
        lo, hi = m_yr.conf_int().loc[term]
        rows.append({"year": yr, "coef": m_yr.params[term], "lo": lo, "hi": hi})
yd = pd.DataFrame(rows).sort_values("year")

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(yd["year"], yd["lo"], yd["hi"], alpha=0.2)
ax.plot(yd["year"], yd["coef"], marker="o", ms=3)
ax.axhline(0, color="grey", lw=1)
ax.set_xlabel("year"); ax.set_ylabel("log rate vs. base year")
ax.set_title("Year effects on the accusation rate (country FE, 95% CI)")
viz.savefig(fig, "h1_year_effects")
plt.show()

### 5d. Classifier threshold

The accusation dataset applies one cut to the lielines score. If the score
*distribution* drifts over time, the trend could be an artefact of that single
cut. Here the rate is rebuilt straight from the corpus at several thresholds.

The first cell checks what `lie_score` actually contains — if it is a hard 0/1
label rather than a probability, this check does not apply and can be skipped.

In [ ]:
print(con.execute("""
    SELECT COUNT(*) AS n,
           MIN(TRY_CAST(lie_score AS DOUBLE)) AS min,
           MAX(TRY_CAST(lie_score AS DOUBLE)) AS max,
           AVG(TRY_CAST(lie_score AS DOUBLE)) AS mean,
           COUNT(DISTINCT lie_score) AS n_distinct
    FROM corpus
""").df().T)

In [ ]:
thr_rows = []
for thr in (0.5, 0.7, 0.9):
    t = con.execute(f"""
        SELECT country,
               CAST(substr(date, 1, 4) AS INT) AS year,
               COUNT(*) AS n_sentences,
               SUM(CASE WHEN TRY_CAST(lie_score AS DOUBLE) > {thr} THEN 1 ELSE 0 END)
                   AS n_accusations
        FROM corpus
        WHERE date IS NOT NULL AND length(date) >= 4
        GROUP BY 1, 2
    """).df()
    t = t[t["n_sentences"] >= MIN_SENTENCES].copy()
    t["year_c"] = t["year"] - cyf["year"].mean()
    m = (smf.glm("n_accusations ~ year_c + C(country)", data=t,
                 family=sm.families.Poisson(),
                 offset=np.log(t["n_sentences"]))
            .fit(cov_type="cluster", cov_kwds={"groups": t["country"]}))
    thr_rows.append((thr, report(m, f"threshold > {thr}")))

print("\nStable coefficients across thresholds => the trend is not an artefact\n"
      "of where the cut was placed.")

## 6. Country-by-country trends — is the null hiding a split?

The section-4 model estimates the *average* within-country trend. If some
countries rise and others fall, that average is near zero even though a great
deal is happening. The small multiples in section 3 suggest exactly that.

Here each country gets its own trend coefficient with a confidence interval,
sorted and plotted. Countries whose interval excludes zero are moving; those
straddling it are genuinely flat.

Short series give unstable slopes, so countries below `MIN_YEARS_TREND` are
dropped.

In [ ]:
MIN_YEARS_TREND = 10     # lower this to include short series (with wide CIs)

rows = []
for c, d in cyf.groupby("country"):
    d = d.sort_values("year")
    if d["year"].nunique() < MIN_YEARS_TREND:
        continue
    try:
        m = (smf.glm("n_accusations ~ year_c", data=d,
                     family=sm.families.NegativeBinomial(alpha=1.0),
                     offset=np.log(d["n_sentences"]))
                .fit(cov_type="HC1"))
        b = m.params["year_c"]
        lo, hi = m.conf_int().loc["year_c"]
        rows.append({
            "country": c, "n_years": d["year"].nunique(),
            "b": b, "p": m.pvalues["year_c"],
            "pct_decade": (np.exp(b * 10) - 1) * 100,
            "lo_decade":  (np.exp(lo * 10) - 1) * 100,
            "hi_decade":  (np.exp(hi * 10) - 1) * 100,
        })
    except Exception as e:
        print(f"  {c}: model failed ({type(e).__name__})")

tr = pd.DataFrame(rows).sort_values("pct_decade").reset_index(drop=True)
tr["direction"] = np.where(tr["lo_decade"] > 0, "up",
                    np.where(tr["hi_decade"] < 0, "down", "flat"))

print(f"{len(tr)} countries with >= {MIN_YEARS_TREND} years\n")
print(tr[["country", "n_years", "pct_decade", "lo_decade", "hi_decade",
          "p", "direction"]].round(1).to_string(index=False))

In [ ]:
colors = {"up": "#c0603f", "down": "#2f6f9f", "flat": "#9a9a9a"}

fig, ax = plt.subplots(figsize=(8, 0.34 * len(tr) + 1.5))
y = np.arange(len(tr))
for i, r in tr.iterrows():
    ax.plot([r["lo_decade"], r["hi_decade"]], [i, i],
            color=colors[r["direction"]], lw=2, solid_capstyle="round")
    ax.plot(r["pct_decade"], i, "o", color=colors[r["direction"]], ms=5)

ax.axvline(0, color="black", lw=1, zorder=0)
ax.set_yticks(y)
ax.set_yticklabels([f"{r.country}  (n={r.n_years})" for r in tr.itertuples()],
                   fontsize=9)
ax.set_xlabel("% change in accusation rate per decade  (95% CI)")
ax.set_title("Country-specific trends in accusations of lying")
ax.margins(y=0.01)

handles = [plt.Line2D([], [], color=v, lw=2,
                      label={"up": "rising", "down": "falling",
                             "flat": "no clear trend"}[k])
           for k, v in colors.items()]
ax.legend(handles=handles, loc="lower right", frameon=True)
fig.tight_layout()
viz.savefig(fig, "h1_country_slopes")
plt.show()

n = tr["direction"].value_counts()
print(f"rising: {n.get('up', 0)}   falling: {n.get('down', 0)}   "
      f"flat: {n.get('flat', 0)}")
print(f"median country trend: {tr['pct_decade'].median():+.1f}% / decade")
print("\nA roughly even up/down split means the section-4 null reflects "
      "cancellation,\nnot stability — report this figure alongside it.")

### Where next

If the split is real, the interesting question stops being *whether* accusations
rose and becomes *why some parliaments diverge*. Two follow-ups worth a look
before the conference:

- do rising countries coincide with populist entry into parliament? — that links
  H1 directly to H2, and the party-level populism scores are already in the panel;
- is the direction related to chamber type or corpus source (Westminster vs.
  continental), which would be a measurement story rather than a political one.

Save `tr` if you want to regress country trends on country-level features later.

## 7. Verdict

Fill in once the cells above have run:

| check | year coef | % / decade |
|---|---|---|
| Poisson, country FE (main) | | |
| Negative binomial | | |
| Balanced panel (>= 15 yrs) | | |
| Source-dataset FE | | |
| Threshold 0.7 / 0.9 | | |
| Countries rising / falling / flat (s.6) | | |

**H1 is supported** if the coefficient is positive, significant, and survives
5a–5d. **H1b is supported** if it sits near zero with a tight CI — note that a
*precise null* is the substantive claim here, so report the CI, not just the
p-value.

**But check section 6 before concluding either.** An average of zero can mean
"nothing changes anywhere" or "as many parliaments rise as fall." Those are very
different claims and only the country-slope figure distinguishes them. If the
split is real, the finding is that there is no *common* trend — which contradicts
H1 without simply confirming H1b.

Watch for the two failure modes: a trend that appears only in the pooled line and
vanishes under country FE (composition), or one that vanishes under source FE
(digitisation / translation).